In [8]:
import sys
import os
import json
import numpy as np
import argparse
import pickle
import inspect


print("Python executable:", sys.executable)
print("Python path:", sys.path)
sys.path.insert(0, "/home/fkrafft")


from kingmaker_fork.likelihood_analysis.utils.trial_runner_config_loading import save_king_trial_runner_config
from kingmaker_fork.likelihood_analysis.utils.parallel_king_trials import get_many_fits_from_trials
from kingmaker_fork.likelihood_analysis.analysis_helpers.to_jsonable import to_jsonable
from kingmaker_fork.kingmaker.wrapper import KingSpatialLikelihood

from  csky_gfu_tests.custom_gfu_specs import GFUDataSpecs as custom_gfu
import csky as cy
from csky.utils import Arrays

timer = cy.timing.Timer()
time = timer.time

Python executable: /home/fkrafft/venvs/icecube-py3v4.4.0/bin/python
Python path: ['/home/fkrafft', '/cvmfs/icecube.opensciencegrid.org/py3-v4.4.0/RHEL_9_x86_64_v2/spack/opt/spack/linux-almalinux9-x86_64_v2/gcc-13.3.0/python-3.12.5-kszr5bufasykf3jernzq4bvwqs7jp4pc/lib/python312.zip', '/cvmfs/icecube.opensciencegrid.org/py3-v4.4.0/RHEL_9_x86_64_v2/spack/opt/spack/linux-almalinux9-x86_64_v2/gcc-13.3.0/python-3.12.5-kszr5bufasykf3jernzq4bvwqs7jp4pc/lib/python3.12', '/cvmfs/icecube.opensciencegrid.org/py3-v4.4.0/RHEL_9_x86_64_v2/spack/opt/spack/linux-almalinux9-x86_64_v2/gcc-13.3.0/python-3.12.5-kszr5bufasykf3jernzq4bvwqs7jp4pc/lib/python3.12/lib-dynload', '', '/home/fkrafft/venvs/icecube-py3v4.4.0/lib/python3.12/site-packages']


In [9]:
ana_dir = cy.utils.ensure_dir('/data/user/fkrafft/updated_king_bias')
seed = 0
mp_cpus = 16
calc_sens = bool(1)
N_trials = 1000
candidate_id = 'E8_b26d8e_D4_fc5cd0_S15_MC300'
out_dir = os.path.join(ana_dir, candidate_id)
if not os.path.exists(out_dir):
    os.mkdir(os.path.join(out_dir, 'log'))

src_sin_dec = 0.0
config_dir = '/home/fkrafft/kingmaker_fork/likelihood_analysis/scripts/bias_candidate_selection.json'


file_identifier = f"SINDEC_{src_sin_dec}_NTRIALS_{N_trials}_{candidate_id}"

In [10]:
with open(config_dir, "r") as f:
            config= json.load(f)
candidate_dir = os.path.join(out_dir, candidate_id)
os.makedirs(candidate_dir, exist_ok=True)

for c in config["candidates"]:
        if c["id"] == candidate_id:
            candidate = c
            break
        
bins_cfg = candidate["parametrization_bins"]
parametrization_bins = {}

#build parametrization bins
for key, value in bins_cfg.items():
    if key == "dec_sindec_edges":
        if isinstance(value, int):
            parametrization_bins["dec"] = value
        else:
            parametrization_bins["dec"] = np.arcsin(np.asarray(value, dtype=float))

    elif key == "log10energy": 
        if isinstance(value, int):
            parametrization_bins["log10energy"] = value
        else:
            parametrization_bins["log10energy"] = np.asarray(value, dtype=float)

    elif key == "sigma":
        parametrization_bins["sigma"] = int(value)

    else:
        parametrization_bins[key] = value

In [11]:
# load spectral indices
spectral_indices= np.asarray(config["spectral_indices"])

#default gamma for signal injection


fixed = config["fixed"]
signal_injection_gamma = fixed["signal_injection_gamma"]
dpsi_nbins=fixed["dpsi_nbins"]
minimum_counts=candidate["minimum_counts"]
weight_field=fixed["weight_field"]
true_ra_name=fixed["true_ra_name"]
true_dec_name=fixed["true_dec_name"]
true_energy_name=fixed["true_energy_name"]
angular_cutoff_deg = fixed["angular_cutoff_deg"]

In [12]:
for path in [
    "/data/user/fkrafft/csky_repo",
    "/data/user/fkrafft/updated_king_bias",
    "/data/ana/PointSource/GFU/online_v001-p10",
]:
    print(
        path,
        "exists=", os.path.exists(path),
        "readable=", os.access(path, os.R_OK),
        "executable=", os.access(path, os.X_OK),
    )

/data/user/fkrafft/csky_repo exists= True readable= True executable= True
/data/user/fkrafft/updated_king_bias exists= True readable= True executable= True
/data/ana/PointSource/GFU/online_v001-p10 exists= True readable= True executable= True


In [15]:
import os

src = "/data/ana/PointSource/GFU/online_v001-p10/gfu_online/version-001-p09/IC86_2023_MC.npy"
dst = "/data/user/fkrafft/csky_repo/gfu_online/version-001-p09/IC86_2023_MC.npy"

print("source exists:", os.path.exists(src))
print("destination exists:", os.path.exists(dst))

source exists: True
destination exists: True


In [16]:
repo_cache = cy.utils.ensure_dir('/data/user/fkrafft/csky_repo')
repo = cy.selections.Repository(
    local_root = repo_cache,
    remote_root= '/data/ana/PointSource/GFU/online_v001-p10')


print("Repository signature:", inspect.signature(cy.selections.Repository))
print("repo =", repo)
for name in ["root", "local_root", "remote_root", "base_dir", "dir"]:
    print(name, "=", getattr(repo, name, None))


with time('ana setup (from cache-to-disk)'):
    ana = cy.get_analysis(repo, 'version-001-p09' , custom_gfu.gfu_19_to_23, dir = ana_dir)
    ana.save(ana_dir)

fit_cache_filename = f"king_fit_custom_gfu_{file_identifier}.npz"

Repository signature: (local_root=None, remote_root=None, username=None)
repo = <csky.selections.Repository object at 0x7eff6fef6390>
root = None
local_root = /data/user/fkrafft/csky_repo
remote_root = /data/ana/PointSource/GFU/online_v001-p10
base_dir = None
dir = None
Setting up Analysis for:
GFU_2019_2023
Setting up GFU_2019_2023...
Reading /data/user/fkrafft/csky_repo/gfu_online/version-001-p09/IC86_2023_MC.npy ...
Reading /data/user/fkrafft/csky_repo/gfu_online/version-001-p09/IC86_2019_data.npy ...
Reading /data/user/fkrafft/csky_repo/gfu_online/version-001-p09/IC86_2020_data.npy ...
Reading /data/user/fkrafft/csky_repo/gfu_online/version-001-p09/IC86_2021_data.npy ...
Reading /data/user/fkrafft/csky_repo/gfu_online/version-001-p09/IC86_2022_data.npy ...
Reading /data/user/fkrafft/csky_repo/gfu_online/version-001-p09/GRL/IC86_2019_data.npy ...
Reading /data/user/fkrafft/csky_repo/gfu_online/version-001-p09/GRL/IC86_2020_data.npy ...
Reading /data/user/fkrafft/csky_repo/gfu_online

In [ ]:
king_wrapper = KingSpatialLikelihood(
                                    signal_events = ana[0].sig.as_array,
                                    parametrization_bins = parametrization_bins,
                                    spectral_indices= spectral_indices,
                                    cache_name = os.path.join(out_dir, fit_cache_filename),
                                    dpsi_nbins=dpsi_nbins,
                                    minimum_counts=minimum_counts,
                                    weight_field = weight_field,
                                    true_ra_name = true_ra_name,
                                    true_dec_name = true_dec_name,
                                    true_energy_name = true_energy_name,
                                    angular_cutoff = np.radians(angular_cutoff_deg),
                                       )


In [ ]:
features = {
    "ra": "ra",
    "dec": "dec",
    "sigma": "sigma",
    "energy": "energy",
}

fits = {
    "gamma": spectral_indices,
}

dtype = [('gamma', '<f8'), ('ns', '<f8'), ('ts', '<f8')]

#src position
ra = 0
dec = np.arcsin(src_sin_dec)


# Configure trial runner
cy.CONF['mp_cpus'] = mp_cpus

srcs = cy.sources(ra, dec)

# run trial runner passing king function evaluation
king_params = dict(angular_cutoff = np.radians(angular_cutoff_deg),
                   spectral_indicies = spectral_indices,
                   parametrization_bins = parametrization_bins,
                   cache_dir = out_dir)

In [ ]:
king_wrapper.parametrization_bins

In [ ]:
tr: cy.trial.TrialRunner = cy.get_trial_runner(ana = ana,
                         src = srcs,
                         flux = cy.hyp.PowerLawFlux(signal_injection_gamma),
                         space = 'king',
                         king_params = king_params,
                         window_dist = king_params.get("angular_cutoff", np.pi),
                         circle_cut = False,
                         use_bdt = False
                         )

In [ ]:
config_path = os.path.join(
    out_dir,
    f"king_trial_runner_config_sindec_{src_sin_dec}_{file_identifier}.pkl",
)

save_king_trial_runner_config(
    config_path,
    out_dir=out_dir,
    file_identifier=file_identifier,
    src_sin_dec=src_sin_dec,
    mp_cpus=mp_cpus,
    gamma=signal_injection_gamma,
    spectral_indices=spectral_indices,
    parametrization_bins=parametrization_bins,
    weight_field=weight_field,
    angular_cutoff_deg=angular_cutoff_deg,
    dpsi_nbins=dpsi_nbins,
    minimum_counts=minimum_counts,
    features=features,
    fits=fits,
)


In [ ]:
bg_trials = []



with time("run test trials"):
    bg_trials = tr.get_many_fits(
        n_trials= 10,
        n_sig=0,
        logging=True,
        mp_cpus=mp_cpus,
        seed=seed,
    )
    new_bg_array = np.zeros(len(bg_trials), dtype=dtype)

    new_bg_array["gamma"] = bg_trials["gamma"]
    new_bg_array["ns"] = bg_trials["ns"]
    new_bg_array["ts"] = bg_trials["ts"]

    bg_dir = cy.utils.ensure_dir(out_dir)

    np.save(
        f'{bg_dir}/king_bkg_trials_sindec_{np.round(np.sin(dec), 3)}_N_{N_trials}_{file_identifier}.npy',
        new_bg_array,
    )
    

In [ ]:
bg_chi2 = cy.dists.Chi2TSD(new_bg_array['ts'])
bg_chi2

In [ ]:
#test for bias
n_sigs = np.r_[:31:3]
trials = [tr.get_many_fits(100, n_sig=n_sig, logging=False, seed=n_sig) for n_sig in n_sigs]
    
#We add the true number of events injected for bookkeeping convenience:
for (n_sig, t) in zip(n_sigs, trials):
    t['ntrue'] = np.repeat(n_sig, len(t))

#Concatenate the trial batches:
allt = cy.utils.Arrays.concatenate(trials)
with open(os.path.join(out_dir, f"king_bias_{src_sin_dec}"
    f"_N_{N_trials}_{file_identifier}.json"), "wb") as f:
    pickle.dump(allt, f)